# Member 1 — CLIP compositionality diagnosis (0-A / 0-B / 0-C)

This notebook is the **readable record of member 1's own contribution**: it loads
the result files committed under `report/results/` and walks through the three
diagnostic probes and the behavioural-space dimensionality reduction. It runs on
CPU with no checkpoints or external data; every number is traceable to a file in
this repository.

**Question:** is CLIP's compositionality failure caused by *missing information*
in the representation, or by a *readout* problem?

- **0-B** — sensitivity to bag-of-words-preserving transformations (readout)
- **0-A** — linear probe on frozen features (information availability)
- **0-C** — failure taxonomy and near-miss analysis on ARO / SugarCrepe++
- **UMAP + KMeans** — behavioural geometry of the similarity responses

> Note: the dimensionality-reduction cells use *score/response vectors*, not
> hidden CLIP embeddings. No feature cache was shipped with the project, so we
> neither fabricate nor claim any latent-space visualisation.

In [ ]:
from pathlib import Path
import json
import csv
import numpy as np
import matplotlib.pyplot as plt

RES = Path("../report/results")
FIG = Path("../report/figures")

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": "#333333", "text.color": "#222222",
    "xtick.color": "#444444", "ytick.color": "#444444", "font.size": 10,
})
print("Result files:", sorted(p.name for p in RES.glob("0*")))

Result files: ['0a_winoground.json', '0b_probe.csv', '0b_summary.json', '0c_examples.md', '0c_summary.json', '0c_taxonomy.csv']


## 1. 0-B — bag-of-words-preserving probes

Five transformations keep the word multiset identical; `unrelated` is a control.
If CLIP truly bound relations, accuracy on the transformations should be well
above the 50% chance level. The metric is `P(score(original) > score(transformed))`.

In [ ]:
b = json.loads((RES / "0b_summary.json").read_text(encoding="utf-8"))
print("model=%s / %s, n_samples=%d" %
      (b["model"], b["pretrained"], b["n_samples"]))
print(f"{'probe':22s} {'n':>6s} {'accuracy':>9s} {'95% CI':>20s} {'mean margin':>12s}")
for p in b["probes"]:
    print(f"{p['probe']:22s} {p['n']:6d} {p['accuracy']:9.3f} "
          f"[{p['ci_low']:.3f}, {p['ci_high']:.3f}]".ljust(61) +
          f"{p['mean_margin']:12.3f}")

model=ViT-B-32 / laion2b_s34b_b79k, n_samples=5000
probe                       n  accuracy               95% CI  mean margin
unrelated                5000     0.995 [0.993, 0.997]              0.228
word_shuffle             4998     0.769 [0.757, 0.780]              0.022
trigram_shuffle          4993     0.605 [0.591, 0.619]              0.007
adj_swap                 1527     0.590 [0.566, 0.615]              0.004
subj_obj_swap            3472     0.582 [0.565, 0.598]              0.005
noun_swap                4947     0.581 [0.568, 0.594]              0.004
relation_reverse         3122     0.568 [0.551, 0.585]              0.003


The 0-B result is also plotted in the main report (`fig_0b.png`). Accuracy on
`unrelated` is ~99.5%, while all five bag-preserving transforms sit at 56–61%,
close to chance. The average margins are only 0.003–0.007 versus 0.228 for the
control: the decisions are made near the boundary, not from a confident
understanding of the relation.

In [ ]:
img = plt.imread(FIG / "fig_0b.png")
plt.figure(figsize=(10, 4))
plt.imshow(img)
plt.axis("off")
plt.title("Fig 2-1 in the report — 0-B probe accuracy (committed figure)")
plt.show()

## 2. 0-A — is the information in the representation?

We build a binary choice (correct binding vs swapped binding) on the 400
Winoground pairs and fit a 5-fold `GroupKFold` linear probe on frozen CLIP
features. The input vector is
`[g0, g1, t0, t1, g0*t0, g0*t1, g1*t0, g1*t1, |g0-g1|, |t0-t1|]`.

If the linear probe substantially exceeds zero-shot group accuracy, the signal
is present but not read out by cosine similarity.

In [ ]:
a = json.loads((RES / "0a_winoground.json").read_text(encoding="utf-8"))
print(json.dumps(a, indent=2, ensure_ascii=False))

{
  "benchmark": "winoground",
  "n_pairs": 400,
  "zero_shot_text": 0.3525,
  "zero_shot_image": 0.1125,
  "zero_shot_group": 0.0775,
  "linear_probe_group": 0.5,
  "linear_probe_std": 0.0,
  "gain_over_group": 0.4225
}


The linear probe reaches 50.0% group accuracy against 7.7% zero-shot — a
+42.2pt gap. This is the key motivation for a lightweight adapter /
hard-negative fine-tuning rather than an architecture replacement. It does
**not** prove training dynamics (the full 0-A training-dynamics variant was
cut for compute reasons); it is inference-time evidence only.

## 3. 0-C — failure taxonomy on ARO / SugarCrepe++

0-C reads the E0 per-sample files and buckets errors by compositional class and
by margin, separating *near misses* (margin within 0.01 of the boundary) from
*confident errors*.

In [ ]:
c = json.loads((RES / "0c_summary.json").read_text(encoding="utf-8"))
print(json.dumps(c, indent=2, ensure_ascii=False)[:4000])
print()   # blank line
print("--- taxonomy table ---")
with (RES / "0c_taxonomy.csv").open(encoding="utf-8") as f:
    for row in csv.reader(f):
        print(" | ".join(row))

{
  "aro": {
    "n": 71679,
    "accuracy": 0.5797234894460023,
    "categories": {
      "attribute": {
        "benchmark": "aro",
        "category": "attribute",
        "n": 28748,
        "errors": 12116,
        "error_rate": 0.4214554055934326,
        "mean_margin": 0.00401683384552598,
        "near_miss_share": 0.5664410696599538,
        "confident_error_share": 0.006437768240343348
      },
      "order": {
        "benchmark": "aro",
        "category": "order",
        "n": 18994,
        "errors": 6069,
        "error_rate": 0.3195219543013583,
        "mean_margin": 0.010238217189908028,
        "near_miss_share": 0.5743944636678201,
        "confident_error_share": 0.004119294776734223
      },
      "relation": {
        "benchmark": "aro",
        "category": "relation",
        "n": 23937,
        "errors": 11940,
        "error_rate": 0.4988093746083469,
        "mean_margin": 3.679479777929373e-05,
        "near_miss_share": 0.5795644891122278,
        "confiden

In [ ]:
img = plt.imread(FIG / "fig_0c.png")
plt.figure(figsize=(10, 4))
plt.imshow(img)
plt.axis("off")
plt.title("Fig 2-2 in the report — 0-C error taxonomy (committed figure)")
plt.show()

Relation and attribute errors dominate, and 62–69% of them are near misses.
That pattern predicts that a small decision-boundary shift can help — which is
exactly what E3 delivers on the `order` subsets, while VG relation/attribute
subsets remain essentially flat.

## 4. Behavioural-space dimensionality reduction (UMAP + KMeans)

Two real response matrices are available in `report/results/persample/`:

1. **Winoground** — the four similarity cells `s00, s01, s10, s11` (n=400).
2. **0-B** — the six per-transform margins on all-six-complete cases (n=2,168).

We reduce each to 2-D with UMAP and run KMeans (k=2) on the original response
vectors. The adjusted Rand index (ARI) against the correctness label tests
whether correct and incorrect cases occupy separable behavioural clusters.

In [ ]:
w = np.load(RES / "persample" / "e0_winoground_persample.npz", allow_pickle=True)
Xw = np.column_stack([w[k].astype(float) for k in ("s00", "s01", "s10", "s11")])
yw = w["group_correct"].astype(int)
print("Winoground response matrix:", Xw.shape,
      "| group-correct rate: %.3f" % yw.mean())
print("First 3 rows:", Xw[:3])

Winoground response matrix: (400, 4) | group-correct rate: 0.077
First 3 rows: [[0.34604108 0.32996321 0.32802665 0.32109848]
 [0.20581147 0.23122776 0.21506685 0.23124108]
 [0.3391239  0.3460443  0.33547676 0.3487303 ]]


In [ ]:
brows = [json.loads(line) for line in
         (RES / "persample" / "0b_persample.jsonl").open(encoding="utf-8")]
keys = ["subj_obj_swap", "noun_swap", "word_shuffle",
        "trigram_shuffle", "relation_reverse", "unrelated"]
brows = [r for r in brows if all(k in r for k in keys)]
Xb = np.array([[r[k]["margin"] for k in keys] for r in brows], dtype=float)
yb = (Xb > 0).all(axis=1).astype(int)
print("0-B margin matrix:", Xb.shape,
      "| all-six-correct rate: %.3f" % yb.mean())

0-B margin matrix: (2168, 6) | all-six-correct rate: 0.236


Run `python scripts/make_umap_figures.py` to regenerate the two committed
figures and the machine-readable cluster summary. Running it is optional here;
the images below are the ones embedded in the final report.

In [ ]:
img = plt.imread(FIG / "fig_umap_winoground.png")
plt.figure(figsize=(12, 5))
plt.imshow(img)
plt.axis("off")
plt.title("Behavioural geometry of E0 Winoground responses")
plt.show()

In [ ]:
img = plt.imread(FIG / "fig_umap_0b.png")
plt.figure(figsize=(12, 5))
plt.imshow(img)
plt.axis("off")
plt.title("Behavioural geometry of 0-B transform margins")
plt.show()

In [ ]:
print((RES / "umap_cluster_summary.md").read_text(encoding="utf-8"))

# 降维/聚类分析说明

本分析使用 UMAP 降维与 KMeans 聚类，输入为评测时保存的**行为特征**（相似度响应 / 探针 margin），不是 CLIP 隐层向量。仓库未附带特征缓存，因此不构造或声称任何 latent embedding 的可视化。

# UMAP / KMeans 聚类摘要（行为特征空间）

> 说明：输入是模型的**分数响应向量**，不是 CLIP 隐层 embedding；仓库未提供特征缓存，因此不伪造 latent vector。

## Winoground（E0，n=400）

- 输入特征：`[s00, s01, s10, s11]` 四个图文相似度单元（4 维）。
- 降维：UMAP(n_neighbors=10, min_dist=0.12, random_state=42) → 2 维。
- KMeans(k=2, seed=42) 与 group 正确性的调整兰德指数 ARI = **0.000**。
- 聚类正确率：C0 9.9%，C1 5.6%。
- 解读：ARI 接近 0 时说明仅凭四个相似度数值无法把对错样例分成两类，与 0-A「信息存在但零样本读数读不出」的结论一致；这不是说表征中完全没有信息（线性探针另见 2.2 节）。

## 0-B 探针（E0，完成全部 6 类变换的 n=2168）

- 输入特征：6 类保持词袋变换的 margin（原句分数 − 变换句分数，6 维）。
- 降维：UMAP(n_neighbors=15, min_dist=0.12, random_state=42) → 2 维。
- KMeans(k=2) 与「六类全部拒绝」标签的 ARI = **-0.004**。
- 至少一类 near-miss（margin∈(−0.01,0]）的样本：1392 (64.2%)。
- 解读：margin 的几何结构连续、对错样本没有干净的簇分离，说明多数失败是决策边界附近的挪动，而不是「完全不知道」；这与 0-C 的近失手统计一致。



**Interpretation.** Both ARIs are essentially zero (0.000 and −0.004), so the
four similarity cells and six transform margins do *not* cleanly separate
correct from incorrect cases in behavioural space. This is consistent with the
0-A/0-C finding: failure is mostly a small boundary shift on compositionally
hard pairs, not a binary presence/absence of the signal. It is deliberately
reported as a negative clustering result rather than being spun as success.

## 5. Traceability

| Notebook section | Source file | Report section |
|---|---|---|
| 0-B | `report/results/0b_summary.json`, `persample/0b_persample.jsonl` | §2.1 / Fig 2-1 |
| 0-A | `report/results/0a_winoground.json` | §2.2 |
| 0-C | `report/results/0c_summary.json`, `0c_taxonomy.csv`, `0c_examples.md` | §2.3 / Fig 2-2 |
| UMAP/KMeans | `persample/e0_winoground_persample.npz`, `scripts/make_umap_figures.py` | §2.4 / Fig 2-3, 2-4 |

All numbers in this notebook are read from committed files. No GPU, API key,
or raw dataset download is required to reproduce the analysis presented here.